In [1]:
# 06d-1. 기본 설정

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)


RIDGE_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "06a_ridge"
)

RF_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "06b_random_forest"
)

XGB_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "06c_xgboost"
)

In [2]:
# 06d-2. fold 결과 불러오기

ridge_results = pd.read_csv(
    RIDGE_DIR
    / "ridge_fold_results.csv"
)

rf_results = pd.read_csv(
    RF_DIR
    / "rf_fold_results.csv"
)

xgb_results = pd.read_csv(
    XGB_DIR
    / "xgb_fold_results.csv"
)


print(
    "ridge:",
    ridge_results.shape
)

print(
    "rf:",
    rf_results.shape
)

print(
    "xgb:",
    xgb_results.shape
)

ridge: (9, 17)
rf: (9, 20)
xgb: (9, 26)


In [3]:
# 06d-3. oos prediction 불러오기

ridge_oos = (
    pq.read_table(
        RIDGE_DIR
        / "ridge_oos_predictions.parquet"
    )
    .to_pandas()
)

rf_oos = (
    pq.read_table(
        RF_DIR
        / "rf_oos_predictions.parquet"
    )
    .to_pandas()
)

xgb_oos = (
    pq.read_table(
        XGB_DIR
        / "xgb_oos_predictions.parquet"
    )
    .to_pandas()
)


print(
    "ridge:",
    ridge_oos.shape
)

print(
    "rf:",
    rf_oos.shape
)

print(
    "xgb:",
    xgb_oos.shape
)

ridge: (11140, 8)
rf: (11140, 8)
xgb: (11140, 8)


In [4]:
# 06d-4. oos 표본 일치 검증

KEY_COLS = [
    "signal_date",
    "execution_date",
    "ticker"
]


for name, data in {
    "ridge": ridge_oos,
    "rf": rf_oos,
    "xgb": xgb_oos
}.items():

    print(
        name,
        "rows:",
        len(data),
        "signals:",
        data["signal_date"].nunique(),
        "duplicates:",
        data[KEY_COLS].duplicated().sum()
    )


ridge_keys = set(
    map(
        tuple,
        ridge_oos[
            KEY_COLS
        ].to_numpy()
    )
)

rf_keys = set(
    map(
        tuple,
        rf_oos[
            KEY_COLS
        ].to_numpy()
    )
)

xgb_keys = set(
    map(
        tuple,
        xgb_oos[
            KEY_COLS
        ].to_numpy()
    )
)


print(
    "ridge == rf:",
    ridge_keys == rf_keys
)

print(
    "ridge == xgb:",
    ridge_keys == xgb_keys
)

print(
    "rf == xgb:",
    rf_keys == xgb_keys
)

ridge rows: 11140 signals: 223 duplicates: 0
rf rows: 11140 signals: 223 duplicates: 0
xgb rows: 11140 signals: 223 duplicates: 0
ridge == rf: True
ridge == xgb: True
rf == xgb: True


In [7]:
%pip install scikit-learn

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----------------------------- ---------- 6.0/8.2 MB 30.9 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 25.8 MB/s  0:00:00
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ------- -------------------------------- 7.1/36.6 MB 34.4 MB/s eta 0:00:01
   -------------- ------------------------- 13.1/36.6 MB 30.3 MB/s eta 0:00:01
   ---------------------- ----------------- 20.2/36.6 MB 31.3 MB/s eta 0:00:01
   ----------------------------- ---------- 27.3/36.6 MB 32.3 MB/s eta 0:00:01
   -------------------------------------- - 34.9/36.6 MB 32.3 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 29.5 MB/s  0:00:01

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   

In [8]:
# 06d-4a. 비교용 설정

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

TARGET = "target_return"


def calculate_ic(
    data,
    prediction_column,
    target_column
):

    ic_values = []

    for _, group in data.groupby(
        "signal_date"
    ):

        if len(group) < 2:
            continue

        if (
            group[prediction_column].nunique() < 2
            or
            group[target_column].nunique() < 2
        ):
            continue

        ic = (
            group[
                prediction_column
            ]
            .rank()
            .corr(
                group[
                    target_column
                ]
                .rank()
            )
        )

        if pd.notna(ic):
            ic_values.append(ic)

    return np.array(
        ic_values
    )


print(
    "target:",
    TARGET
)

target: target_return


In [9]:
# 06d-5. target 일치 검증

comparison_check = (
    ridge_oos[
        KEY_COLS
        + [
            TARGET,
            "dummy_prediction"
        ]
    ]
    .rename(
        columns={
            TARGET: "ridge_target",
            "dummy_prediction":
                "ridge_dummy"
        }
    )
    .merge(
        rf_oos[
            KEY_COLS
            + [
                TARGET,
                "dummy_prediction"
            ]
        ]
        .rename(
            columns={
                TARGET: "rf_target",
                "dummy_prediction":
                    "rf_dummy"
            }
        ),
        on=KEY_COLS,
        how="inner"
    )
    .merge(
        xgb_oos[
            KEY_COLS
            + [
                TARGET,
                "dummy_prediction"
            ]
        ]
        .rename(
            columns={
                TARGET: "xgb_target",
                "dummy_prediction":
                    "xgb_dummy"
            }
        ),
        on=KEY_COLS,
        how="inner"
    )
)


print(
    "merged rows:",
    len(comparison_check)
)

print(
    "target ridge-rf:",
    np.allclose(
        comparison_check[
            "ridge_target"
        ],
        comparison_check[
            "rf_target"
        ]
    )
)

print(
    "target ridge-xgb:",
    np.allclose(
        comparison_check[
            "ridge_target"
        ],
        comparison_check[
            "xgb_target"
        ]
    )
)

print(
    "dummy ridge-rf:",
    np.allclose(
        comparison_check[
            "ridge_dummy"
        ],
        comparison_check[
            "rf_dummy"
        ]
    )
)

print(
    "dummy ridge-xgb:",
    np.allclose(
        comparison_check[
            "ridge_dummy"
        ],
        comparison_check[
            "xgb_dummy"
        ]
    )
)

merged rows: 11140
target ridge-rf: True
target ridge-xgb: True
dummy ridge-rf: True
dummy ridge-xgb: True


In [10]:
# 06d-6. model prediction 결합

model_oos_comparison = (
    ridge_oos[
        KEY_COLS
        + [
            "name",
            TARGET,
            "dummy_prediction",
            "prediction",
            "fold"
        ]
    ]
    .rename(
        columns={
            "prediction":
                "ridge_prediction"
        }
    )
    .merge(
        rf_oos[
            KEY_COLS
            + [
                "prediction"
            ]
        ]
        .rename(
            columns={
                "prediction":
                    "rf_prediction"
            }
        ),
        on=KEY_COLS,
        how="inner"
    )
    .merge(
        xgb_oos[
            KEY_COLS
            + [
                "prediction"
            ]
        ]
        .rename(
            columns={
                "prediction":
                    "xgb_prediction"
            }
        ),
        on=KEY_COLS,
        how="inner"
    )
    .sort_values(
        [
            "signal_date",
            "ticker"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "shape:",
    model_oos_comparison.shape
)

print(
    "signals:",
    model_oos_comparison[
        "signal_date"
    ].nunique()
)

print(
    model_oos_comparison.head()
)

shape: (11140, 10)
signals: 223
  signal_date execution_date  ticker    name  target_return  dummy_prediction  \
0  2021-11-05     2021-11-08  000270      기아      -0.022441          0.001886   
1  2021-11-05     2021-11-08  000660  SK하이닉스       0.014117          0.001886   
2  2021-11-05     2021-11-08  003490    대한항공       0.003254          0.001886   
3  2021-11-05     2021-11-08  003670  포스코케미칼      -0.046495          0.001886   
4  2021-11-05     2021-11-08  005380     현대차      -0.030328          0.001886   

   ridge_prediction  fold  rf_prediction  xgb_prediction  
0          0.004952     1       0.000708        0.003759  
1          0.005220     1       0.000708        0.003759  
2          0.004782     1       0.000708        0.003759  
3          0.005201     1       0.000708        0.003072  
4          0.004928     1       0.000708        0.003759  


In [12]:
# 06d-7. pooled model 성능 비교
# sanity check
# sanity check = 결과가 상식적이고 기존 계산과 일치하는지 확인하는 검증

def evaluate_oos_model(
    data,
    prediction_column
):

    y_true = data[
        TARGET
    ]

    y_pred = data[
        prediction_column
    ]


    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )


    ic_values = calculate_ic(
        data,
        prediction_column,
        TARGET
    )


    prediction_std = (
        data
        .groupby(
            "signal_date"
        )[
            prediction_column
        ]
        .std()
        .mean()
    )


    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "mean_ic": (
            ic_values.mean()
            if len(ic_values) > 0
            else np.nan
        ),
        "median_ic": (
            np.median(
                ic_values
            )
            if len(ic_values) > 0
            else np.nan
        ),
        "ic_signals":
            len(ic_values),
        "prediction_std":
            prediction_std
    }


model_summary = pd.DataFrame(
    [
        {
            "model": "dummy_mean",
            **evaluate_oos_model(
                model_oos_comparison,
                "dummy_prediction"
            )
        },
        {
            "model": "ridge",
            **evaluate_oos_model(
                model_oos_comparison,
                "ridge_prediction"
            )
        },
        {
            "model": "random_forest",
            **evaluate_oos_model(
                model_oos_comparison,
                "rf_prediction"
            )
        },
        {
            "model": "xgboost",
            **evaluate_oos_model(
                model_oos_comparison,
                "xgb_prediction"
            )
        }
    ]
)


print(
    model_summary.to_string(
        index=False
    )
)

        model     rmse      mae        r2   mean_ic  median_ic  ic_signals  prediction_std
   dummy_mean 0.078617 0.054508 -0.000844       NaN        NaN           0        0.000000
        ridge 0.078607 0.054452 -0.000598 -0.068794  -0.076495         223        0.000242
random_forest 0.079609 0.055463 -0.026252 -0.024128  -0.007155         189        0.000707
      xgboost 0.079507 0.055300 -0.023635 -0.013659  -0.015127         150        0.003630


In [13]:
# 06d-8. fold별 model 비교

fold_comparison = (
    ridge_results[
        [
            "fold",
            "ridge_rmse",
            "ridge_mae",
            "ridge_r2",
            "mean_ic",
            "dummy_rmse"
        ]
    ]
    .rename(
        columns={
            "mean_ic":
                "ridge_mean_ic"
        }
    )
    .merge(
        rf_results[
            [
                "fold",
                "rf_rmse",
                "rf_mae",
                "rf_r2",
                "mean_ic"
            ]
        ]
        .rename(
            columns={
                "mean_ic":
                    "rf_mean_ic"
            }
        ),
        on="fold",
        how="inner"
    )
    .merge(
        xgb_results[
            [
                "fold",
                "xgb_rmse",
                "xgb_mae",
                "xgb_r2",
                "mean_ic"
            ]
        ]
        .rename(
            columns={
                "mean_ic":
                    "xgb_mean_ic"
            }
        ),
        on="fold",
        how="inner"
    )
)


fold_comparison[
    "best_rmse_model"
] = (
    fold_comparison[
        [
            "ridge_rmse",
            "rf_rmse",
            "xgb_rmse"
        ]
    ]
    .idxmin(
        axis=1
    )
    .str.replace(
        "_rmse",
        "",
        regex=False
    )
)


print(
    fold_comparison.to_string(
        index=False
    )
)

 fold  ridge_rmse  ridge_mae  ridge_r2  ridge_mean_ic  dummy_rmse  rf_rmse   rf_mae     rf_r2  rf_mean_ic  xgb_rmse  xgb_mae    xgb_r2  xgb_mean_ic best_rmse_model
    1    0.064162   0.047195 -0.013011      -0.057940    0.064548 0.065774 0.048468 -0.064538   -0.039894  0.064984 0.047729 -0.039140     0.054452           ridge
    2    0.065222   0.047433 -0.018258      -0.072346    0.065276 0.069515 0.051486 -0.156714   -0.052400  0.070949 0.051634 -0.204925    -0.080671           ridge
    3    0.074754   0.049259 -0.016305      -0.045032    0.074470 0.075019 0.049453 -0.023524         NaN  0.074454 0.049034 -0.008158          NaN             xgb
    4    0.076923   0.050796 -0.008782      -0.092479    0.076923 0.077697 0.051675 -0.029194    0.019594  0.078070 0.051970 -0.039095     0.028355           ridge
    5    0.076069   0.051544 -0.005062      -0.096273    0.076069 0.076433 0.051720 -0.014705   -0.012920  0.076114 0.051519 -0.006263          NaN           ridge
    6    0.08182

In [14]:
# 06d-9. comparison 결과 저장

COMPARISON_DIR = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "06d_comparison"
)

COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True
)

model_summary.to_csv(
    COMPARISON_DIR
    / "model_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

fold_comparison.to_csv(
    COMPARISON_DIR
    / "fold_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("saved")

saved
